# Q1. Embedding a query
Embed the following query:

How does approximate nearest neighbor search work?

The embedder returns a vector of 384 numbers. What's the first value (v[0])?

In [1]:
from embedder import Embedder

embed = Embedder()

q1 = "How does approximate nearest neighbor search work?"


v1 = embed.encode(q1)
v1[0]

np.float64(-0.02058203437252893)

**Answer:** -0.02

# Loading the data
We pull the lesson pages from the course repository, the same way as in homework 1. We pin to commit 8c1834d so everyone works with the same data.

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

# Q2. Cosine similarity
The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.

Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?

In [12]:
file = '02-vector-search/lessons/07-sqlitesearch-vector.md'

resultados = [doc['content'] for doc in documents if doc['filename'] == file]

content = resultados[0] if resultados else "File not found"

cv = embed.encode(content)

print(cv.dot(v1))

0.36107027225589694


**Answer:** 0.37

# Q3. Chunking and search by hand
A full page covers several topics, which waters down its embedding.

We chunk the pages the same way as in homework 1:

from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

We embed every chunk's content with encode_batch, stack the vectors into a matrix X, and score the Q1 query against all chunks:

scores = X.dot(v)

Which file does the highest-scoring chunk belong to (its filename)?

In [38]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

chunks_content = [chunk["content"] for chunk in chunks]

X = embed.encode_batch(chunks_content)

scores = X.dot(v1)

print(chunks[scores.argmax()]["filename"])

02-vector-search/lessons/07-sqlitesearch-vector.md


**Answer:** 02-vector-search/lessons/07-sqlitesearch-vector.md

# Q4. Vector search with minsearch
We've done vector search by hand, which is good for learning, but it's not what we do in practice. In practice we use libraries.

Let's use VectorSearch from minsearch and run a search for the following query:

What metric do we use to evaluate a search engine?

Which file is the filename of the first result?

In [51]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, chunks)

query = "What metric do we use to evaluate a search engine?"
query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=5)

print(results[0]["filename"])

04-evaluation/lessons/05-search-metrics.md


**Answer:** 04-evaluation/lessons/05-search-metrics.md

# Q5. Text search vs vector search
Vector search matches by meaning, keyword search by exact words.

Let's compare them. Index the same chunks with Index from minsearch. Use content as a text field.

Run both searches for this query:

How do I store vectors in PostgreSQL?

Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?